# Exotic Rates

**Start here:** This deep dive expands on `02_pricing/pricing_across_asset_classes.ipynb`; use `02_pricing/pricing_fundamentals.ipynb` for the shared instrument JSON -> market -> model pipeline.

**Purpose:** Demonstrate the deterministic coupon and payoff helpers behind several exotic rate structures.

**Prerequisites:** Familiarity with coupon mechanics, path-dependent products, and CMS-style rate references.

**In this notebook:** We inspect coupon profiles for a TARN, a snowball, and an inverse floater, then compute a CMS spread option intrinsic payoff and a callable range-accrual coupon.


## Concept

These helpers focus on deterministic payoff logic rather than full Monte Carlo pricing. That makes them especially helpful for:

1. Understanding how coupon state evolves path by path.
2. Sanity-checking intrinsic payoff formulas.
3. Building intuition before using the full pricing pipeline.


In [ ]:
import numpy as np

from finstack_quant.valuations import (
    callable_range_accrual_accrued,
    cms_spread_option_intrinsic,
    inverse_floater_coupon_profile,
    snowball_coupon_profile,
    tarn_coupon_profile,
)

rng = np.random.default_rng(seed=42)


## Coupon profiles and intrinsic payoffs

The next cell reuses the script inputs directly so you can inspect how each product responds to a simple path or fixing set. Notebook format works well here because each printed block is effectively a worked example.


In [ ]:
tarn = tarn_coupon_profile(
    fixed_rate=0.08,
    coupon_floor=0.0,
    floating_fixings=np.linspace(0.06, 0.015, 10).tolist(),
    target_coupon=0.20,
    day_count_fraction=0.5,
)
print("TARN coupons (%)      :", [round(100 * x, 3) for x in tarn["coupons_paid"]])
print("TARN cumulative (%)   :", [round(100 * x, 3) for x in tarn["cumulative"]])
print("TARN redemption index :", tarn["redemption_index"])

snowball = snowball_coupon_profile(
    initial_coupon=0.05,
    fixed_rate=0.06,
    floating_fixings=np.linspace(0.05, 0.01, 8).tolist(),
    floor=0.0,
    cap=float("inf"),
)
print("\nSnowball coupons (%) :", [round(100 * x, 3) for x in snowball])

inverse = inverse_floater_coupon_profile(
    fixed_rate=0.08,
    floating_fixings=np.linspace(0.01, 0.06, 8).tolist(),
    floor=0.0,
    cap=float("inf"),
    leverage=2.0,
)
print("Inverse floater (%)   :", [round(100 * x, 3) for x in inverse])

cms_payoff = cms_spread_option_intrinsic(
    long_cms=0.045,
    short_cms=0.033,
    strike=0.005,
    is_call=True,
    notional=10_000_000.0,
)
print(f"\nCMS spread option intrinsic: ${cms_payoff:,.2f}")

observations = rng.normal(loc=0.035, scale=0.01, size=250).tolist()
range_accrual = callable_range_accrual_accrued(
    lower=0.02,
    upper=0.05,
    observations=observations,
    coupon_rate=0.05,
    day_count_fraction=250.0 / 360.0,
)
in_range = sum(1 for obs in observations if 0.02 <= obs <= 0.05)
print(f"Range-accrual in-range observations: {in_range} / {len(observations)}")
print(f"Range-accrual coupon (% notional) : {100 * range_accrual:.3f}%")


## Takeaways

- The deterministic helpers are ideal for understanding product mechanics before you bring in model or market assumptions.
- `tarn_coupon_profile()` and `snowball_coupon_profile()` make path dependence explicit.
- `cms_spread_option_intrinsic()` and `callable_range_accrual_accrued()` are convenient sanity checks for structured-rate payoff logic.


In [ ]:
{
    "tarn_redeemed_early": tarn["redeemed_early"],
    "snowball_last_coupon": round(snowball[-1], 6),
    "inverse_last_coupon": round(inverse[-1], 6),
    "cms_intrinsic": round(cms_payoff, 2),
    "range_accrual": round(range_accrual, 6),
}


## Bermudan swaption: explicit schedule and fitted Hull–White inputs

Use the supported `bermudan_swaption` JSON type for an explicit multi-date exercise schedule. A `Swaption` example provides canonical underlying-leg specifications; setting `exercise_style="bermudan"` on an ordinary swaption does not supply this schedule. Model keys come from `list_models_grouped()`.

The fixture uses a 2027-01-15 to 2032-01-15 swap and two exercise dates. The final contractual payment is a business day, so it remains within the model horizon after business-day adjustment. The 3% discount curve and the explicit $(\kappa,\sigma)=(0.05,0.01)$ Hull–White pair are illustrative controlled inputs, not a claimed fit to live quotes. No volatility surface is silently calibrated during pricing.

The Monte Carlo engine applies LSMC with stochastic discounting and exposes price, standard error, interval, path budget and seed. Its default regression uses the simulated sample; it does not expose the equity helper's independent `pricing_seed` interface. Compare to the Hull–White tree under the same model assumptions as a numerical cross-check. The printed interval measures sampling uncertainty; it is not a bound on policy or time-grid error.


In [ ]:
import datetime as dt
import json
import math
from finstack_quant.core.market_data import DiscountCurve, MarketContext
from finstack_quant.valuations.instruments import Swaption, list_models_grouped, price_instrument

underlying = json.loads(Swaption.example_bermudan().to_json())["instrument"]["spec"]
for leg_name in ("underlying_fixed_leg", "underlying_float_leg"):
    underlying[leg_name]["start"] = "2027-01-15"
    underlying[leg_name]["end"] = "2032-01-15"
underlying["underlying_fixed_leg"]["rate"] = "0.03"
bermudan_spec = {
    "id": "ANALYST-BERMUDAN", "option_type": "call", "settlement": "physical",
    "notional": {"amount": "10000000", "currency": "USD"},
    "underlying_fixed_leg": underlying["underlying_fixed_leg"],
    "underlying_float_leg": underlying["underlying_float_leg"],
    "bermudan_schedule": {"exercise_dates": ["2028-01-18", "2029-01-16"],
                          "lockout_end": None, "notice_days": 0},
    "bermudan_type": "co_terminal", "vol_surface_id": "USD-SWPNVOL", "attributes": {},
    "instrument_pricing_overrides": {"model_config": {
        "hw1f_mean_reversion": 0.05, "hw1f_sigma": 0.01, "mc_paths": 8192, "tree_steps": 160}},
}
bermudan_json = json.dumps({"schema": "finstack_quant.instrument/1",
                           "instrument": {"type": "bermudan_swaption", "spec": bermudan_spec}})
bermudan_market = MarketContext().insert(DiscountCurve.flat("USD-OIS", dt.date(2026, 1, 15), 0.03))
assert "monte_carlo_hull_white_1f" in list_models_grouped()["bermudan_swaption"]
mc_bermudan = price_instrument(bermudan_json, bermudan_market, "2026-01-15", "monte_carlo_hull_white_1f")
tree_bermudan = price_instrument(bermudan_json, bermudan_market, "2026-01-15", "hull_white_1f")
mc_repeat = price_instrument(bermudan_json, bermudan_market, "2026-01-15", "monte_carlo_hull_white_1f")
mc_standard_error = mc_bermudan.get_metric("mc_stderr")
assert mc_bermudan.price == mc_repeat.price
assert mc_bermudan.get_metric("lsmc_num_paths") == 8192
assert math.isfinite(mc_standard_error) and mc_standard_error > 0
assert abs(mc_bermudan.price - tree_bermudan.price) < 4 * mc_standard_error
print(f"HW1F tree PV={tree_bermudan.price:,.2f}; HW1F LSMC PV={mc_bermudan.price:,.2f}")
for metric in ("mc_stderr", "lsmc_num_paths", "lsmc_seed", "lsmc_ci95_low", "lsmc_ci95_high"):
    print(f"{metric}: {mc_bermudan.get_metric(metric):,.6f}")
